# Ekstraksi Fitur Warna (RGB Mean, Std, Skewness) dan Tekstur (GLCM) serta Label

Notebook ini mengekstrak fitur visual lengkap dari setiap citra pada folder **For Classification**:
1. **Fitur Warna (Color Moments orde 1, 2, dan 3)**:
   - `R_mean`, `G_mean`, `B_mean` (Rata-rata intensitas warna per kanal)
   - `R_std`, `G_std`, `B_std` (Standar deviasi variasi warna per kanal)
   - `R_skew`, `G_skew`, `B_skew` (Skewness kemencengan distribusi warna per kanal)
   - `gray_skew` (Skewness pada citra keabuan/grayscale)
2. **Fitur Tekstur GLCM (Gray-Level Co-occurrence Matrix Haralick)**:
   - `glcm_contrast`: Kontras variasi intensitas lokal antar pasangan piksel
   - `glcm_dissimilarity`: Perbedaan nilai tingkat keabuan
   - `glcm_homogeneity`: Keseragaman lokal distribusi matriks
   - `glcm_energy`: Keseragaman tekstur (akar dari ASM)
   - `glcm_correlation`: Korelasi ketergantungan spasial nilai piksel
   - `glcm_asm`: Angular Second Moment

Label diambil dari folder kelas **`Rapi`**, **`Berantakan`**, dan **`Kosong`** di dalam folder `For Classification`.
Hasilnya disimpan sebagai `fitur_citra_lengkap.csv` dan `fitur_rgb_std.csv`.

## 1. Import library dan tentukan folder data

Path gambar yang diklasifikasikan berada di dalam folder `For Classification` yang memuat subfolder `Rapi`, `Berantakan`, dan `Kosong`.
Jika notebook dan folder `For Classification` berada di direktori yang sama, konfigurasi otomatis menggunakan folder kerja.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "For Classification"

if not DATASET_DIR.is_dir():
    DATASET_DIR = Path(r"C:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra") / "For Classification"

NAMA_FOLDER_KELAS = ("Berantakan", "Kosong", "Rapi")

if not DATASET_DIR.is_dir() or not all((DATASET_DIR / nama).is_dir() for nama in NAMA_FOLDER_KELAS):
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan di: {DATASET_DIR}. Pastikan folder Berantakan, Kosong, dan Rapi ada di dalamnya."
    )

print(f"Folder dataset (For Classification): {DATASET_DIR}")
print(f"Kelas yang didukung: {NAMA_FOLDER_KELAS}")

Folder dataset (For Classification): C:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra\For Classification
Kelas yang didukung: ('Berantakan', 'Kosong', 'Rapi')


## 2. Ekstraksi Fitur Warna dan Tekstur GLCM (Kelas Rapi, Berantakan, Kosong)

Citra diubah ke **RGB** untuk menghitung Mean, Standar Deviasi, dan Skewness tiap kanal. Citra juga diubah ke **Grayscale** untuk menghitung matriks GLCM 4 arah (0 derajat, 45 derajat, 90 derajat, 135 derajat) dengan jarak d=1 dan mengekstrak 6 fitur statistik Haralick untuk masing-masing kelas (`rapi`, `berantakan`, dan `kosong`).

In [2]:
def extract_glcm_features(gray):
    """
    Menghitung GLCM 4 arah (0, 45, 90, 135 derajat) jarak d=1
    dan mengekstrak 6 fitur tekstur Haralick utama.
    """
    levels = 256
    glcm = np.zeros(levels * levels, dtype=np.int32)

    # 0 derajat (horizontal)
    glcm += np.bincount(((gray[:, :-1].astype(np.int32) << 8) | gray[:, 1:]).ravel(), minlength=65536)
    # 90 derajat (vertikal)
    glcm += np.bincount(((gray[:-1, :].astype(np.int32) << 8) | gray[1:, :]).ravel(), minlength=65536)
    # 45 derajat (diagonal kanan)
    glcm += np.bincount(((gray[:-1, :-1].astype(np.int32) << 8) | gray[1:, 1:]).ravel(), minlength=65536)
    # 135 derajat (diagonal kiri)
    glcm += np.bincount(((gray[:-1, 1:].astype(np.int32) << 8) | gray[1:, :-1]).ravel(), minlength=65536)

    # Jadikan simetris & normalisasi
    glcm = glcm.reshape((levels, levels)).astype(np.float64)
    glcm += glcm.T
    s = glcm.sum()
    if s > 0:
        glcm /= s

    # Ekstraksi fitur statistik Haralick
    i, j = np.indices((levels, levels), dtype=np.float64)
    contrast = float(np.sum(glcm * ((i - j) ** 2)))
    dissimilarity = float(np.sum(glcm * np.abs(i - j)))
    homogeneity = float(np.sum(glcm / (1.0 + (i - j) ** 2)))
    asm = float(np.sum(glcm ** 2))
    energy = float(np.sqrt(asm))

    mu_i = np.sum(i * glcm)
    mu_j = np.sum(j * glcm)
    var_i = np.sum(glcm * ((i - mu_i) ** 2))
    var_j = np.sum(glcm * ((j - mu_j) ** 2))
    std_prod = np.sqrt(var_i * var_j)
    correlation = float(np.sum(glcm * (i - mu_i) * (j - mu_j)) / std_prod if std_prod > 0 else 1.0)

    return {
        "glcm_contrast": contrast,
        "glcm_dissimilarity": dissimilarity,
        "glcm_homogeneity": homogeneity,
        "glcm_energy": energy,
        "glcm_correlation": correlation,
        "glcm_asm": asm,
    }


LABEL_MAP = {
    "Rapi": "rapi",
    "Berantakan": "berantakan",
    "Kosong": "kosong",
}
EKSTENSI_GAMBAR = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

NAMA_FITUR_WARNA = [
    "R_mean", "G_mean", "B_mean",
    "R_std", "G_std", "B_std",
    "R_skew", "G_skew", "B_skew",
    "gray_skew"
]
NAMA_FITUR_GLCM = [
    "glcm_contrast", "glcm_dissimilarity", "glcm_homogeneity",
    "glcm_energy", "glcm_correlation", "glcm_asm"
]
NAMA_FITUR = [*NAMA_FITUR_WARNA, *NAMA_FITUR_GLCM]

baris_hasil = []

for nama_folder, label in LABEL_MAP.items():
    folder_kelas = DATASET_DIR / nama_folder
    if not folder_kelas.is_dir():
        raise FileNotFoundError(f"Folder kelas tidak ditemukan: {folder_kelas}")

    daftar_gambar = sorted(
        path for path in folder_kelas.rglob("*")
        if path.is_file() and path.suffix.lower() in EKSTENSI_GAMBAR
    )

    for path_gambar in daftar_gambar:
        with Image.open(path_gambar) as gambar:
            rgb = np.asarray(gambar.convert("RGB"), dtype=np.float32)
            gray = np.asarray(gambar.convert("L"), dtype=np.uint8)

        # 1. Fitur Warna: Mean dan Std
        mean_rgb = rgb.mean(axis=(0, 1))
        std_rgb = rgb.std(axis=(0, 1), ddof=0)

        # 2. Fitur Skewness: Warna (RGB) dan Keabuan (Grayscale)
        diff_rgb = rgb - mean_rgb
        std_safe = np.where(std_rgb == 0, 1e-10, std_rgb)
        skew_rgb = np.mean(diff_rgb ** 3, axis=(0, 1)) / (std_safe ** 3)

        diff_gray = gray.astype(np.float32) - gray.mean()
        std_gray = gray.std()
        skew_gray = float(np.mean(diff_gray ** 3) / (std_gray ** 3) if std_gray > 0 else 0.0)

        # 3. Fitur Tekstur GLCM
        glcm_feats = extract_glcm_features(gray)

        baris_hasil.append({
            "file_name": path_gambar.name,
            "label": label,
            "R_mean": mean_rgb[0],
            "G_mean": mean_rgb[1],
            "B_mean": mean_rgb[2],
            "R_std": std_rgb[0],
            "G_std": std_rgb[1],
            "B_std": std_rgb[2],
            "R_skew": skew_rgb[0],
            "G_skew": skew_rgb[1],
            "B_skew": skew_rgb[2],
            "gray_skew": skew_gray,
            **glcm_feats,
        })

if not baris_hasil:
    raise ValueError(f"Tidak ada gambar yang didukung di {DATASET_DIR}")

df_fitur = pd.DataFrame(baris_hasil, columns=["file_name", "label", *NAMA_FITUR])
print(f"Total gambar berhasil diekstrak: {len(df_fitur)}")
print(f"Total kolom fitur: {len(NAMA_FITUR)}")
display(df_fitur.head())

Total gambar berhasil diekstrak: 245
Total kolom fitur: 16


,file_name,label,R_mean,G_mean,B_mean,R_std,G_std,B_std,R_skew,G_skew,B_skew,gray_skew,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation,glcm_asm
0,IMG-20261105-WA0052.jpg,rapi,115.922516,115.263855,108.083150,50.596110,56.355553,63.412247,0.267057,0.187625,0.036101,-0.007178,61.337570,2.787149,0.533626,0.035841,0.988874,0.001285
1,IMG-20261105-WA0110.jpg,rapi,106.447460,103.808420,102.343000,56.783990,67.670540,68.218650,0.201589,0.206170,0.165386,0.235651,228.535188,10.640404,0.113326,0.012989,0.972051,0.000169
2,IMG-20261108-WA0005.jpg,rapi,133.887190,132.574520,129.505280,69.461780,70.083680,71.092060,-0.362826,-0.249410,-0.019462,-0.294189,169.758153,6.579182,0.305140,0.018922,0.982294,0.000358
3,IMG-20261108-WA0006.jpg,rapi,151.563960,152.696910,151.400710,58.094456,59.934067,65.895860,-0.768609,-0.746395,-0.734284,-0.765781,164.649147,5.053656,0.454082,0.032928,0.975516,0.001084
4,IMG-20261109-WA0025.jpeg,rapi,128.475780,129.721560,126.583496,52.227160,60.273647,65.082504,-0.151009,-0.193221,-0.410312,-0.146324,59.773633,3.167272,0.451198,0.031922,0.990779,0.001019


## 3. Periksa label dan simpan hasil

Dataset kini mencakup 3 kelas: `rapi`, `berantakan`, dan `kosong` dari folder `For Classification`.
Kolom fitur berjumlah 16 (10 fitur warna/intensitas + 6 fitur tekstur GLCM). Hasil disimpan ke `fitur_citra_lengkap.csv` dan `fitur_rgb_std.csv`.

In [4]:
print("Jumlah gambar per label:")
display(df_fitur["label"].value_counts().rename_axis("label").to_frame("jumlah"))

kolom_csv = ["file_name", "label", *NAMA_FITUR]

# Simpan ke fitur_citra_lengkap.csv dan perbarui fitur_rgb_std.csv
file_output_lengkap = BASE_DIR / "fitur_citra_lengkap.csv"
df_fitur[kolom_csv].to_csv(file_output_lengkap, index=False)

file_output_lama = BASE_DIR / "fitur_rgb_std.csv"
df_fitur[kolom_csv].to_csv(file_output_lama, index=False)

print(f"\nCSV berhasil disimpan:")
print(f"- {file_output_lengkap}")
print(f"- {file_output_lama}")
print(f"Jumlah total kolom: {len(kolom_csv)} (2 metadata + {len(NAMA_FITUR)} fitur)")

Jumlah gambar per label:


,jumlah
label,
berantakan,110
rapi,93
kosong,42



CSV berhasil disimpan:
- c:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra\fitur_citra_lengkap.csv
- c:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra\fitur_rgb_std.csv
Jumlah total kolom: 18 (2 metadata + 16 fitur)
